In [ ]:
from pathlib import Path
import numpy as np
from split_dataset import SplitDataset
import json
import flammkuchen as fl

In [ ]:
master = Path(r"Z:\Hagar\e0075\v04_4x4")
all_fish = list(master.glob("*_f*"))
print(all_fish)

In [ ]:
for f in all_fish:
    print(f)
    if not (f / "original_int").exists():
        inc_factor = 2**12
        metadata_file_stack = f / "original/stack_metadata.json"
        with open(str(metadata_file_stack), "r") as fr:
            stack_param = json.load(fr)
        n_z, nx, ny = stack_param["shape_full"][1:4]
        stack = SplitDataset(f / "original/")
        print(n_z)
        for i in range(n_z):
            if i < 10:
                file_name = '000' + str(i) + '.h5'
            else:
                file_name = '00' + str(i) + '.h5'
            print(file_name)

            data_in = stack[:, i, :, :]
            print(np.max(data_in), np.min(data_in), np.mean(data_in))

            data_out = (data_in + 0.05) * inc_factor
            print(np.max(data_out), np.min(data_out), np.mean(data_out))
            print(np.percentile(data_in, 99.999))

            data_out = data_out.astype(np.uint16)
            data_out = np.expand_dims(data_out, 1)

            orig_dir = (f / "original_int")
            orig_dir.mkdir(exist_ok=True)

            data_out_folder = str(f / "original_int" / file_name)

            fl.save(data_out_folder, {"stack_4D": data_out}, compression="blosc")
            data_in = 0
            data_out = 0
